In [5]:
import pandas as pd
import joblib

In [6]:
model = joblib.load("../models/weekly_model_v4_promotions.pkl")

In [7]:
df_new = pd.read_csv("../data/processed/weekly_sales_v4_promotions.csv")

df_new["week_start"] = pd.to_datetime(df_new["week_start"])
df_new.head()

,year,week,store_id,product_id,units_sold,avg_price,avg_regular_price,avg_discount_pct,promo_intensity,avg_starting_inventory,...,black_friday_week,month_calendar,season,promo_days_in_week,avg_weekly_discount_pct,has_promo_week,promo_type_Bundle,promo_type_Discount,promo_type_Free Accessory,promo_type_Price Slash
0,2020,53,1,1001,2,445838.0,445838.0,0.0,0.0,9.666667,...,0,1,Dry,0.0,0.0,0.0,0,0,0,0
1,2021,1,1,1001,7,445838.0,445838.0,0.0,0.0,6.000000,...,0,1,Dry,0.0,0.0,0.0,0,0,0,0
2,2021,2,1,1001,6,445838.0,445838.0,0.0,0.0,7.571429,...,0,1,Dry,0.0,0.0,0.0,0,0,0,0
3,2021,3,1,1001,5,445838.0,445838.0,0.0,0.0,5.000000,...,0,1,Dry,0.0,0.0,0.0,0,0,0,0
4,2021,4,1,1001,8,445838.0,445838.0,0.0,0.0,5.571429,...,0,1,Dry,0.0,0.0,0.0,0,0,0,0


In [8]:
df_new["store_id"] = df_new["store_id"].astype(str)
df_new["product_id"] = pd.to_numeric(df_new["product_id"], errors="coerce").astype("Int64")

df_new = df_new.dropna(subset=["store_id", "product_id", "units_sold"])
df_new["product_id"] = df_new["product_id"].astype(int)

In [9]:
df_new = df_new.sort_values(["store_id", "product_id", "week_start"]).copy()

group_cols = ["store_id", "product_id"]

df_new["lag_1"] = df_new.groupby(group_cols)["units_sold"].shift(1)
df_new["lag_4"] = df_new.groupby(group_cols)["units_sold"].shift(4)

df_new["rolling_mean_4"] = (
    df_new.groupby(group_cols)["units_sold"]
    .shift(1)
    .rolling(4)
    .mean()
)

df_new = df_new.dropna(subset=["lag_1", "lag_4", "rolling_mean_4"]).copy()

In [10]:
df_new_encoded = pd.get_dummies(
    df_new,
    columns=["season", "store_id", "product_id"],
    drop_first=True
)